# Week 11b.1: Long-Term Memory

Everything our agents have remembered so far lives in a thread. The checkpointer saves the conversation under a `thread_id`, and a new `thread_id` starts with nothing. That is short-term memory.

**Long-term memory** is information that outlasts the thread: a customer's history, a user's preferences, a decision that was made last week and should affect what happens today.

For any long-term memory, there are three design questions:

- **The write path.** What gets written, when, and by whom?
- **The read path.** How does a memory get back into a later decision? Is it looked up by a key, such as a customer id, or found by similarity to the current situation?
- **Where it lives.** Does it survive a restart?

We'll build a memory for the Husky Tech support agent that answers all three. In this notebook, the graph reads and writes the memory on every turn. Deep agents keep the same kind of memory in files, as section 2 of W09b.1 showed.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
keys = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY"]
print("API keys found:", [k for k in keys if os.getenv(k)] or "none (fine with Ollama)")

## 0. Model Setup

Execute one of the following to define the model.

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="claude-haiku-5-5")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. Memory that ends with the thread

First, let's confirm the problem. The support agent below is the W04b agent with a checkpointer. We tell it something on one thread and ask about it on another.

In [ ]:
from langchain.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

import support_tools as st


def tool_calls_of(messages):
    """The tool calls in a run, in order, as (tool, first argument)."""
    calls = []
    for m in messages:
        for call in getattr(m, "tool_calls", None) or []:
            first = next(iter(call["args"].values()), "")
            calls.append((call["name"], str(first)[:60]))
    return calls


def build_support(prompt, tools, store=None):
    model_with_tools = model.bind_tools(tools)
    sys_msg = SystemMessage(content=prompt)

    def assistant(state: MessagesState) -> dict:
        return {"messages": [model_with_tools.invoke([sys_msg] + state["messages"])]}

    builder = StateGraph(MessagesState)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant")
    agent = builder.compile(checkpointer=InMemorySaver())
    return agent


def chat(agent, text, thread, customer_id=None):
    """One turn on a thread. Prints the tool calls made during this turn and the reply.
    We'll use customer_id from section 3 on."""
    config = {"configurable": {"thread_id": thread, "customer_id": customer_id}}
    before = len(agent.get_state(config).values.get("messages", []))
    out = agent.invoke({"messages": [HumanMessage(content=text)]}, config)
    print(f"[{thread}] user:  {text}")
    for name, arg in tool_calls_of(out["messages"][before:]):
        print(f"           tool:  {name}({arg})")
    print(f"[{thread}] agent: {out['messages'][-1].text}\n")
    return out


agent = build_support(st.SYSTEM_PROMPT, st.TOOLS)


In [ ]:
_ = chat(agent, "Hi, I'm customer C-17. Please always reply to me in Spanish.", "monday")

In [ ]:
_ = chat(agent, "Hi, I'm customer C-17. What are your support hours?", "friday")

On the second thread, the agent answers in English. The request from Monday is saved, but only under the `monday` thread, and the agent on `friday` has no way to reach it.

## 2. The store

LangGraph's **store** holds memories outside any thread. In W09b.1, the deep agent's `/memories/` files were kept in a store through `StoreBackend`. Today, we'll use the store directly.

When we save a memory to the store, we give three things:

- The **namespace**, a tuple that works like a folder, such as `("customers", "C-17")`.
- The **key**, which works like a file name, such as `"profile"`.
- The **value**, a dictionary, which works like the contents of the file.

The picture shows a store with memories for two customers. C-17 has one memory, and C-22 has two.

![What the store saves](figures/memory_store.png)

`InMemoryStore` keeps everything in this Python process.

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

In [ ]:
namespace = ("customers", "C-17")
key = "profile"
value = {"memory": "Prefers replies in Spanish."}

`put` saves a value under a namespace and a key:

In [ ]:
#TODO

`search` returns the memories in a namespace, as a list:

In [ ]:
memories = store.search(#TODO)
memories

In [ ]:
store.put(namespace, "device", {"device": "27-inch monitor"})


In [ ]:
c17 = store.search(("customers", "C-17"), limit=100)
c17

In [ ]:
store.list_namespaces(prefix=("customers",))

Each item in the list holds the namespace, the key, the value, and when the memory was created and last updated.

In [ ]:
print(memories[0].key, memories[0].value)

`get` returns one memory, by its namespace and key:

In [ ]:
store.get(namespace, "profile").dict()

A `put` with a key that already exists replaces the old value.

In [ ]:
store.put(namespace, "profile", {"memory": "Prefers replies in Spanish. Owns a 27-inch monitor."})

In [ ]:
store.get(namespace, "profile").value

## 3. The support agent with long term memory

Now let's fix the problem from section 1. We'll keep one memory per customer, a short profile under the key `"profile"`, and add two steps to our default ReAct agent.

![How the graph uses the store](figures/memory_graph.png)


- **Read.** Before the model answers, the assistant node gets the customer's profile from the store and adds it to the system prompt.
- **Write.** On every turn, a `write_memory` node asks the model to update the profile with anything new the customer has said, and saves the result under the same key.

Both nodes need to know which customer is talking. We'll pass the customer's id in the config, next to the `thread_id`:

```python
config = {"configurable": {"thread_id": "friday", "customer_id": "C-17"}}
```

The `thread_id` picks the conversation, and the `customer_id` picks the customer's namespace in the store. The `chat` function from section 1 already takes a `customer_id` for this.

Here are the two prompts.

In [ ]:
ASSISTANT_PROMPT = st.SYSTEM_PROMPT + """

Here is what we know about this customer from earlier conversations (it may be empty).
Follow any preferences it lists.
{memory}"""

WRITE_MEMORY_PROMPT = """You keep a short profile of a Husky Tech customer, to personalize future conversations.

CURRENT PROFILE:
{memory}

Read the customer's messages below and update the profile with anything new the customer has told us about
themselves, such as preferences, the devices they own, or problems they have had.
- Write the profile as a short bulleted list.
- If something new conflicts with the current profile, keep the newer version.
- Include only what the customer said. Do not guess.

Return only the updated profile."""

Next, the nodes. A node can take two more arguments besides the state. LangGraph passes the config we give to `invoke` as `config`, and the store we give to `compile` as `store`.

`get_profile` is a helper that both nodes use. 

`write_memory` sends the model only the customer's messages, because the profile should hold only what the customer said.

In [ ]:
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore

model_with_tools = model.bind_tools(st.TOOLS)


def get_profile(config, store):
    """The customer's profile from the store, or a note that there is none."""
    customer_id = config["configurable"]["customer_id"]
    item = store.get(("customers", customer_id), "profile")
    return item.value["memory"] if item else "No profile yet."


def write_memory(state: MessagesState, config: RunnableConfig, store: BaseStore) -> dict:
    """Update the customer's profile from the conversation, and save it."""

    memory_prompt = WRITE_MEMORY_PROMPT.format(memory=get_profile(config, store))

    customer_messages = [m for m in state["messages"] if m.type == "human"]

    memory_message = [SystemMessage(content=memory_prompt)] + customer_messages
    profile = model.invoke(memory_message)

    customer_id = config["configurable"]["customer_id"]
    store.put(("customers", customer_id), "profile", {"memory": profile.text})

    return {}

In [ ]:
def assistant(state: MessagesState, config: RunnableConfig, store: BaseStore) -> dict:
    """ReAct assistant, with the customer's profile in the system prompt."""
    sys_msg = SystemMessage(content=ASSISTANT_PROMPT.format(memory=get_profile(config, store)))
    return {"messages": [model_with_tools.invoke([sys_msg] + state["messages"])]}

`write_memory` returns an empty dictionary because it does not change the messages. Its only effect is the write to the store.

`write_memory` is added at the start, so the profile is updated before the assistant reads it. The store goes to `compile`, next to the checkpointer. The checkpointer keeps each thread's messages, as before, and the store keeps the profiles across threads.

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("write_memory", write_memory)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(st.TOOLS))
builder.add_edge(START, "write_memory")
builder.add_edge("write_memory", "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

store = InMemoryStore()
support = builder.compile(checkpointer=InMemorySaver(), #TODO)

In [ ]:
from IPython.display import Image, display

display(Image(support.get_graph().draw_mermaid_png()))


We started from an empty store, so the profile from section 2 is gone.

### Monday

C-17 makes the same request as in section 1, this time with a `customer_id`.

In [ ]:
_ = chat(support, "Hi, I'm customer C-17. Please always reply to me in Spanish.", "monday", customer_id="C-17")

Here is C-17's profile after Monday.

In [ ]:
print(store.get(("customers", "C-17"), "profile").value["memory"])

The profile is saved under C-17's namespace, not under the `monday` thread.

### Friday

Here is the question from section 1 again, on a new thread.

In [ ]:
_ = chat(support, "Hi, I'm customer C-17. What are your support hours?", "friday", customer_id="C-17")

This time the agent answers in Spanish. The `friday` thread starts with no messages, as before, but the assistant read C-17's profile from the store before it called the model.

Let's ask the same question for a customer with no profile.

In [ ]:
_ = chat(support, "Hi, I'm customer C-22. What are your support hours?", "friday-c22", customer_id="C-22")

C-22 gets the answer in English. The two questions are the same, and the replies differ only because of what is in the store. This is a good test for any long-term memory: name a decision today that would come out differently without something stored last week.

Notice that `write_memory` calls the model on every turn, even when the customer says nothing new about themselves

We'll build this graph again with other stores in sections 4 and 5. Here is the whole graph in a helper that takes the store as an argument, as `build_support` took the checkpointer in W11a.2.

In [ ]:
def build_memory_support(store):
    """The support agent compiled with the given store."""
    builder = StateGraph(MessagesState)
    builder.add_node("write_memory", write_memory)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(st.TOOLS))
    builder.add_edge(START, "write_memory")
    builder.add_edge("write_memory", "assistant")
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant")
    return builder.compile(checkpointer=InMemorySaver(), store=store)

## 4. Where memory lives

`InMemorySaver` and `InMemoryStore` keep everything in the Python process. Every memory will be gone upon restart or shutdown. 

LangGraph has store and checkpointer classes for SQLite and Postgres with the same interface. Here is the SQLite store, connected the way an app connects. The app opens the file once, when it starts, and builds the agent once with the store.


In [ ]:
import sqlite3
from langgraph.store.sqlite import SqliteStore

# once, when the app starts
connection = sqlite3.connect("husky_memory.db", check_same_thread=False, isolation_level=None)
sqlite_store = SqliteStore(connection)
sqlite_store.setup()          # creates the tables if they are missing

In [ ]:
support_sqlite = build_memory_support(sqlite_store)

- `check_same_thread=False` lets the agent use the connection from the threads that run its tools, as in W11a.2.
- `isolation_level=None` saves each write as soon as it is made. The store needs this setting.

### Monday

In [ ]:
_ = chat(support_sqlite, "Hi, I'm customer C-17. Please always reply to me in Spanish.", "sqlite-monday", customer_id="C-17")

In [ ]:
# the app stops
connection.close()

### Friday, after a restart

Now the app stops and starts again. It closes the connection, opens a new one on the same file, and builds a new agent. Nothing from Monday is left in the process.

In [ ]:
# the app starts again
connection = sqlite3.connect("husky_memory.db", check_same_thread=False, isolation_level=None)
sqlite_store = SqliteStore(connection)
support_sqlite = build_memory_support(sqlite_store)

In [ ]:
_ = chat(support_sqlite, "Hi, I'm customer C-17. What are your support hours?", "sqlite-friday", customer_id="C-17")

The agent read C-17's profile from the file and answered in Spanish. The memory survived because it is in the file, not in the process. The checkpointer has a SQLite version too, `SqliteSaver`, so the conversations in a thread can survive a restart in the same way.

When the app stops for good, it closes the connection.

In [ ]:
connection.close()

## 5. A database in the cloud: Supabase

A SQLite file lives on one machine. An agent deployed on a server, or several copies of it, needs a database that all of them can reach. In W11a.2 you saved threads in Supabase with `PostgresSaver`, and you added the connection string to your `.env` as `SUPABASE_DB_URL`. `PostgresStore` is the store for Postgres. It comes in the same package and has the same interface as the other stores, so the memory moves to the cloud and the agent does not change.

We'll connect the way a deployed app does, The app opens a pool of connections once, when it starts, and compiles the graph once. Every request uses the same agent, and the pool lends it a connection for as long as the request takes.

In [ ]:
from psycopg_pool import ConnectionPool
from langgraph.store.postgres import PostgresStore

load_dotenv(override=True)        # reads SUPABASE_DB_URL even if you added it after starting the kernel
DB_URL = os.getenv("SUPABASE_DB_URL")

# once, when the app starts
pool = ConnectionPool(DB_URL, kwargs={"autocommit": True, "prepare_threshold": 0}, open=True)
pool.wait(timeout=10)        # stop with an error if the database can't be reached
cloud_store = PostgresStore(pool)
cloud_store.setup()          # creates the tables if they are missing

In [ ]:
support_cloud = build_memory_support(cloud_store)

The pool settings are the ones from W11a.2. `PostgresStore` takes the pool the same way `PostgresSaver` did, and the graph is the one from section 3. Only the store changed.

### Monday

In [ ]:
_ = chat(support_cloud, "Hi, I'm customer C-1. Please always reply to me in Spanish.", "cloud-monday", customer_id="C-1")

Open your project in Supabase and go to the **Table Editor**. The `store` table has a row for C-1's profile. The `prefix` column holds the namespace, `customers.C-1`, the `key` column holds `profile`, and `value` holds the profile itself.

### Friday, after a restart

Here is a new agent, built with `build_memory_support`, as if the app had been restarted or a second copy of it were answering. It has not seen Monday's conversation, and it shares only the database.

In [ ]:
support_restarted = build_memory_support(PostgresStore(pool))
_ = chat(support_restarted, "Hi, I'm customer C-1. What are your support hours?", "cloud-friday", customer_id="C-1")


### One security step

As with the checkpointer's tables in W11a.2, turn on **row level security** for the tables the store created. In Supabase, open the **SQL Editor** and run:

```sql
alter table store enable row level security;
alter table store_migrations enable row level security;
```

### Closing the pool

When the app stops, it closes the pool, so the database does not keep its connections open.

In [ ]:
pool.close()

## 6. Things to try

1. On a new thread, have C-17 say "English is fine for me now." Print C-17's profile. Did `write_memory` replace the old preference, keep both, or do something else? Then ask about the support hours on another new thread.

2. Have a longer conversation with a new customer, and print the profile after each turn. Does `write_memory` add anything the customer did not say?

3. LangGraph's own memory example runs `write_memory` after the model answers, instead of before. Change the graph to do the same: start at `assistant`, and when the assistant is done, go to `write_memory` and then to `END`. You can do this with `builder.add_conditional_edges("assistant", tools_condition, {"tools": "tools", END: "write_memory"})`. What changes in C-17's reply on Monday?

In [ ]:
# your experiments here